# 02 · Aggregations: COUNT, SUM, AVG, MIN, MAX, GROUP BY, HAVING

**Dataset:** 1,000 synthetic healthcare claims (January–June 2026). All data is fake.

**Tables:** `claims`, `patients`, `providers`, `procedures`.

> 🇪🇸 **Para Josué:** hoy toca **resumir** muchos claims en pocos números (totales, promedios, conteos), como una tabla dinámica de Excel.
> Hay **4 tablas** (las ves abajo). La primera vez, corre todo con **Run All**. **Guarda con Ctrl + S después de cada ejercicio.**

In [1]:
# Load the CSV files as tables.
import duckdb

duckdb.sql("CREATE OR REPLACE TABLE claims     AS SELECT * FROM '../data/claims.csv'")
duckdb.sql("CREATE OR REPLACE TABLE patients   AS SELECT * FROM '../data/patients.csv'")
duckdb.sql("CREATE OR REPLACE TABLE providers  AS SELECT * FROM '../data/providers.csv'")
duckdb.sql("CREATE OR REPLACE TABLE procedures AS SELECT * FROM '../data/procedures.csv'")
print("Tables loaded: claims, patients, providers, procedures")

Tables loaded: claims, patients, providers, procedures


## The 4 tables

> 🇪🇸 `claims` y `patients` son largas: muestro solo las **primeras 10** filas con `LIMIT 10`.

### `claims` (1,000 rows)

In [2]:
duckdb.sql("""
    SELECT * FROM claims LIMIT 10
""").df()

,claim_id,patient_id,provider_id,date_of_service,cpt_code,billed_amount,paid_amount,claim_status,denial_reason
0,CL00001,PA0106,PR011,2026-01-01,71046,337,239,Paid,NaN
1,CL00002,PA0048,PR003,2026-01-01,93306,1015,850,Paid,NaN
2,CL00003,PA0044,PR004,2026-01-01,97140,163,0,Denied,Service not covered
3,CL00004,PA0141,PR010,2026-01-01,99203,204,119,Paid,NaN
4,CL00005,PA0132,PR009,2026-01-01,97110,180,109,Paid,NaN
5,CL00006,PA0108,PR008,2026-01-01,93306,969,580,Paid,NaN
6,CL00007,PA0040,PR005,2026-01-01,85025,59,35,Paid,NaN
7,CL00008,PA0102,PR005,2026-01-02,99214,212,117,Paid,NaN
8,CL00009,PA0177,PR005,2026-01-02,80053,92,66,Paid,NaN
9,CL00010,PA0091,PR009,2026-01-02,97140,156,127,Paid,NaN


### `patients` (200 rows)

In [3]:
duckdb.sql("""
    SELECT * FROM patients LIMIT 10
""").df()

,patient_id,patient_name,date_of_birth,insurance_plan,state
0,PA0001,Laura Lee,1969-01-19,Commercial,TX
1,PA0002,Michael Lopez,1946-07-01,Commercial,AZ
2,PA0003,Susan Johnson,1945-03-15,Commercial,TX
3,PA0004,Jessica Williams,1978-11-26,Medicaid,AZ
4,PA0005,Linda Davis,1958-01-31,Medicare,AZ
5,PA0006,Laura Johnson,1945-07-20,Medicaid,FL
6,PA0007,David Martinez,1944-03-06,Medicaid,NY
7,PA0008,Maria Moore,1952-12-09,Medicaid,FL
8,PA0009,Richard Lopez,1949-03-30,Medicaid,TX
9,PA0010,Patricia Martin,1989-02-20,Commercial,FL


### `providers` (13 rows)

In [4]:
duckdb.sql("""
    SELECT * FROM providers
""").df()

,provider_id,provider_name,specialty,state
0,PR001,Sunrise Family Clinic,Family Medicine,TX
1,PR002,Lakeside Imaging,Radiology,FL
2,PR003,Northside Cardiology,Cardiology,TX
3,PR004,Bayview Physical Therapy,Physical Therapy,CA
4,PR005,Pine Valley Medical Group,Internal Medicine,AZ
5,PR006,Riverside Family Practice,Family Medicine,FL
6,PR007,Desert Sun Radiology,Radiology,AZ
7,PR008,Golden Coast Heart Center,Cardiology,CA
8,PR009,Harbor Rehab & Therapy,Physical Therapy,FL
9,PR010,Maple Street Clinic,Family Medicine,NY


### `procedures` (13 rows)

In [5]:
duckdb.sql("""
    SELECT * FROM procedures
""").df()

,cpt_code,description,service_type,standard_charge
0,99213,"Office visit, established patient, low complexity",Office Visit,150
1,99214,"Office visit, established patient, moderate co...",Office Visit,220
2,99215,"Office visit, established patient, high comple...",Office Visit,300
3,99203,"Office visit, new patient, low complexity",Office Visit,200
4,71046,"Chest X-ray, 2 views",Imaging,320
5,72148,MRI lumbar spine without contrast,Imaging,1200
6,74177,CT abdomen and pelvis with contrast,Imaging,1500
7,93000,"Electrocardiogram (ECG), complete",Cardiology,210
8,93306,"Echocardiogram, complete",Cardiology,950
9,97110,"Therapeutic exercises, 15 minutes",Physical Therapy,180


---
## Example · GROUP BY

How many claims and how much money billed for each `claim_status`.

> 🇪🇸 **Te recuerdo:** `GROUP BY` **junta** las filas que tienen el mismo valor (aquí, el mismo `claim_status`) y las funciones `COUNT`, `SUM`, `AVG`, `MIN` y `MAX` resumen cada grupo.
> Todo lo que está en el `SELECT` y no es una de esas funciones tiene que ir en el `GROUP BY`.
> `AS` le pone nombre a la columna nueva.

In [6]:
duckdb.sql("""
    SELECT claim_status,
           COUNT(*) AS total_claims,
           SUM(billed_amount) AS total_billed
    FROM claims
    GROUP BY claim_status
    ORDER BY total_claims DESC
""").df()

,claim_status,total_claims,total_billed
0,Paid,763,304037.0
1,Denied,174,97500.0
2,Pending,63,23548.0


> 🇪🇸 Léelo así: 763 claims pagados por $304,037; 174 rechazados por $97,500; 63 pendientes por $23,548.

## Exercise 1 · Overall KPIs

Show **one row** with the totals of the whole dataset.

Columns: `total_claims` (number of claims), `total_billed` (sum of `billed_amount`) and `total_paid` (sum of `paid_amount`).

✔ **1 row:** 1,000 claims · $425,085 billed · $194,761 paid.

> 🇪🇸 Los **KPIs** (*Key Performance Indicators*, indicadores clave) del semestre en una sola fila.
> Como quieres el total de **todo**, no hay grupos: esta vez **no** lleva `GROUP BY`.

In [16]:
duckdb.sql("""
    SELECT 
    COUNT(*) AS total_claims,
    SUM (billed_amount) AS total_billed,
    SUM (paid_amount) AS total_paid
    FROM claims
""").df()

,total_claims,total_billed,total_paid
0,1000,425085.0,194761.0


## Exercise 2 · Billing by provider

For each `provider_id`, show the number of claims and the total billed, from the highest to the lowest total billed.

Columns: `provider_id`, `total_claims`, `total_billed`.

✔ **14 rows.** The first one is **PR002** (85 claims, $96,838).

> 🇪🇸 Igual que el ejemplo, pero agrupando por `provider_id` y ordenando por `total_billed` de mayor a menor.

In [17]:
duckdb.sql("""
    SELECT 
    provider_id,
    COUNT(*) AS total_claims,
    SUM(billed_amount) AS total_billed
    FROM claims
    GROUP BY provider_id 
    ORDER BY total_billed DESC
""").df()

,provider_id,total_claims,total_billed
0,PR002,85,96838.0
1,PR011,87,93578.0
2,PR007,72,75206.0
3,PR008,74,32008.0
4,PR003,79,29421.0
5,PR009,108,18844.0
6,PR004,89,15466.0
7,PR012,83,12979.0
8,PR006,86,12921.0
9,PR005,85,11885.0


## Exercise 3 · Charges by CPT code

For each `cpt_code`, show the number of claims, the **average** billed amount (rounded to 2 decimals), the **lowest** and the **highest** billed amount. Sort by the average, from highest to lowest.

Columns: `cpt_code`, `total_claims`, `avg_billed`, `min_billed`, `max_billed`.

✔ **13 rows.** The first one is **74177**: 90 claims · avg 1539.3 · min 1427 · max 1650.

> 🇪🇸 Cuatro funciones en el mismo `SELECT`: `COUNT`, `AVG`, `MIN` y `MAX`.
> **Te recuerdo** cómo redondear: `ROUND(AVG(billed_amount), 2)` = promedio con 2 decimales.

In [19]:
duckdb.sql("""
    SELECT 
    cpt_code,
    COUNT(*) AS total_claims,
    ROUND(AVG(billed_amount), 2) AS avg_billed,
    ROUND(MIN(billed_amount), 2) AS min_billed,
    ROUND(MAX(billed_amount), 2) AS max_billed
    FROM claims
    GROUP BY cpt_code
    ORDER BY avg_billed DESC
""").df()

,cpt_code,total_claims,avg_billed,min_billed,max_billed
0,74177,90,1539.30,1427,1650
1,72148,87,1234.17,1142,1319
2,93306,37,973.51,907,1044
3,71046,70,326.16,304,352
4,99215,67,307.84,285,330
5,99214,76,224.01,209,242
6,93000,43,219.35,200,231
7,99203,61,204.36,190,219
8,97110,100,183.56,171,198
9,97140,98,164.69,152,175


## Exercise 4 · Money lost by denial reason

Only for **denied** claims: for each `denial_reason`, show how many claims were denied and how much money they billed. Sort by the money, from highest to lowest.

Columns: `denial_reason`, `denied_claims`, `billed_denied`.

✔ **5 rows.** The first one is **Missing prior authorization** (70 claims, $61,028).

> 🇪🇸 Primero `WHERE` (quedarte solo con los `'Denied'`), después `GROUP BY` por motivo.
> El orden de las partes es siempre: `SELECT` → `FROM` → `WHERE` → `GROUP BY` → `ORDER BY`.

In [23]:
duckdb.sql("""
    SELECT 
    denial_reason,
    COUNT(*) AS denied_claims,
    SUM(billed_amount) as billed_denied
    FROM claims
    WHERE claim_status = 'Denied'
    GROUP BY denial_reason
    ORDER BY billed_denied DESC
""").df()

,denial_reason,denied_claims,billed_denied
0,Missing prior authorization,70,61028.0
1,Service not covered,42,20038.0
2,Patient not eligible,30,9513.0
3,Missing information,22,4295.0
4,Duplicate claim,10,2626.0


## Exercise 5 · High-volume providers

Show only the providers with **85 claims or more**, from the most to the fewest claims.

Columns: `provider_id`, `total_claims`.

✔ **6 rows.** The first one is **PR009** (108 claims).

> 🇪🇸 **Te recuerdo:** `WHERE` filtra **filas** (datos que ya están en la tabla); `HAVING` filtra **grupos** (datos calculados, como `COUNT(*)`). Va después del `GROUP BY`:
> ```sql
> GROUP BY provider_id
> HAVING COUNT(*) >= 85
> ```

In [36]:
duckdb.sql("""
    SELECT 
    provider_id,
    COUNT(*) AS total_claims
    FROM claims
    GROUP BY provider_id
    HAVING total_claims > 84
    ORDER BY total_claims DESC
""").df()

,provider_id,total_claims
0,PR009,108
1,PR004,89
2,PR011,87
3,PR006,86
4,PR002,85
5,PR005,85


---
## New: COUNT(DISTINCT ...)

`COUNT(DISTINCT column)` counts how many **different** values a column has.

> 🇪🇸 **Lo nuevo:** `COUNT(*)` cuenta **filas**; `COUNT(DISTINCT patient_id)` cuenta **pacientes distintos** (si un paciente tiene 5 claims, cuenta 1).
> En Excel es como "Quitar duplicados" en una columna y después contar.

**Example:** 1,000 claims, but how many different patients?

In [ ]:
duckdb.sql("""
    SELECT COUNT(*) AS total_claims,
           COUNT(DISTINCT patient_id) AS unique_patients
    FROM claims
""").df()

,total_claims,unique_patients
0,1000,200


> 🇪🇸 1,000 claims de solo 200 pacientes: en promedio, 5 claims por paciente.

## Exercise 6 · Unique patients by provider

For each `provider_id`, show the number of claims and the number of **different** patients, from the most to the fewest patients.

Columns: `provider_id`, `total_claims`, `unique_patients`.

✔ **14 rows.** The first one is **PR009** (108 claims, 80 patients).

> 🇪🇸 Junta el `GROUP BY` del ejercicio 2 con el `COUNT(DISTINCT ...)` del ejemplo.

In [35]:
duckdb.sql("""
    SELECT 
    provider_id,
    COUNT(*) AS total_claims,
    COUNT(DISTINCT patient_id) AS unique_patients
    FROM claims
    GROUP BY provider_id
    ORDER BY unique_patients DESC
""").df()

,provider_id,total_claims,unique_patients
0,PR009,108,80
1,PR011,87,76
2,PR004,89,72
3,PR006,86,71
4,PR012,83,69
5,PR002,85,68
6,PR008,74,67
7,PR005,85,65
8,PR007,72,65
9,PR003,79,62


---
## Check your understanding

> 🇪🇸 Responde en el chat, con tus palabras:
> 1. La tabla `providers` tiene **13** proveedores, pero en el ejercicio 2 te salieron **14** `provider_id`. ¿Cuáles sobran y por qué? (Pista: compara tu resultado con la tabla `providers`. ¿Falta alguno también?)
> 2. En el ejercicio 6, PR009 tiene 108 claims pero 80 pacientes. ¿Por qué los números son distintos?